# Lab 04 — K-Nearest Neighbors on Iris
**Classification Track** · Beginner · ~40 min · 🟢 Colab only

## Scenario
Open the lesson narrative in `lab-steps.html` (same folder) for the full teaching text. This notebook is the **executable lab**: lesson notes as Markdown cells, runnable code as code cells, working against the dataset in this folder.

## You will learn
1. Understand distance-based classification
2. Scale features before KNN
3. Select k via cross-validation
4. Classify a hand-entered flower

## Datasets (this folder)
- `iris.csv`

## How to run on Google Colab
1. Click **Start Lab** — or open the hosted notebook directly: [Open in Colab](https://colab.research.google.com/github/matheshcp/ai_course_content/blob/main/course-02-classical-machine-learning/labs/lab-04-knn-iris/lab-04-knn-iris.ipynb) — it opens under *your* Google account (Colab auto-saves a copy to your Drive; no per-student setup, no Drive API create).
2. Run **Cell 0** first — it downloads `dataset.zip` with wget, unzips it, and every code cell below reads those unzipped files.
3. **Runtime → Run all** (GPU not required for Course 2).
4. Work the **Exercises** cells before revealing **Solutions**.

> Direct-open flow: `Start Lab` → hosted URL → Cell 0 (`wget dataset.zip` + `unzip`) → `Runtime → Run all`.


### Setup (dataset)

Run the next cell (Cell 0) once: it downloads `dataset.zip` with wget and unzips it next to the notebook. All code below reads these unzipped files (`iris.csv`). Skips the download when the files already exist.


In [ ]:
# Cell 0 — dataset first: wget dataset.zip + unzip (run this cell first).
import os, shutil, subprocess, urllib.request, zipfile

LAB_ID = "lab-04-knn-iris"
DATASET_ZIP_URL = "https://raw.githubusercontent.com/matheshcp/ai_course_content/main/course-02-classical-machine-learning/labs/lab-04-knn-iris/bundle/dataset.zip"
NEED = ["iris.csv"]  # unzipped files used by the code below

def _have_files():
    return all(os.path.exists(f) for f in NEED)

def _wget_zip(url, dest):
    # shell equivalent: !wget -q <url> -O dataset.zip
    if shutil.which("wget"):
        subprocess.run(["wget", "-q", url, "-O", dest], check=True)
    else:  # plain Python without wget: stdlib fallback
        urllib.request.urlretrieve(url, dest)

if _have_files():
    print("dataset ready:", ", ".join(NEED))
else:
    _wget_zip(DATASET_ZIP_URL, "dataset.zip")
    # shell equivalent: !unzip -o -q dataset.zip
    with zipfile.ZipFile("dataset.zip") as z:
        z.extractall(".")
    print("downloaded + unzipped dataset.zip ->", ", ".join(NEED))


## Beginner: The Flower Shop That Learns by "Nearest Neighbors"

> **Scenario:** You run a flower shop's online ordering desk. Customers upload a photo, and your app must guess the iris species from four measurements (sepal length/width, petal length/width) so it can route the order to the right grower. You have 150 labeled flowers. Instead of writing rules, you let the app "look at the most similar flowers it has seen" — this is the k-Nearest Neighbors (KNN) idea.
>
> **You will learn:** how distance metrics define "similarity", why feature scaling can make or break a distance-based model, how to pick k with cross-validation, and how to read a decision boundary.
> **Time:** ~40 minutes. **Level:** Beginner. **Needs:** pandas + sklearn. **Env:** 🟢 Colab only.

### Mental map

| Concept | One-line idea | Where it shows up |
|---|---|---|
| Distance metric | How we measure "how far apart" two flowers are | Euclidean vs Manhattan |
| Feature scaling | Putting all measurements on the same scale | StandardScaler |
| k (neighbors) | How many nearby flowers vote on the species | k=1 is noisy, huge k is blurry |
| Cross-validation | Honest estimate of accuracy on unseen flowers | 5-fold CV curve |
| Decision boundary | The map of "which region is which species" | 2D contour plot |

---

## 1. How KNN thinks: "tell me who your neighbors are"

Every recommendation system you have ever used — "customers who bought this also bought…" — is KNN wearing a costume. It is the simplest idea in machine learning: to understand something new, look at the things most like it. If you can define "like", you can build a classifier with zero training.

KNN has no training step in the usual sense. To classify a new flower, it measures the distance from that flower to every flower in the training set, finds the **k closest** ones, and lets them vote. The majority species wins. The only real choice (beyond k) is **how you measure distance**.

Think of each flower as a point in a 4-dimensional space — one axis per measurement. Two flowers that are close in this space have similar sepals and petals, so they are probably the same species. "Close" sounds obvious, but you must pick a geometry, and different geometries give different answers.

The two most common metrics:

- **Euclidean** — straight-line distance: `sqrt(sum((a - b)^2))`. Think "as the crow flies".
- **Manhattan** — sum of absolute differences: `sum(|a - b|)`. Think "city blocks" in a grid-like city.

Euclidean penalizes one large gap more than several small ones (because of the squaring); Manhattan treats every gap equally. In low dimensions they usually agree; in high dimensions they drift apart. On iris, both work well — the species form tight, well-separated clusters.

The snippet below loads the iris data, then trains two KNN classifiers — one Euclidean, one Manhattan — each wrapped in a pipeline with a StandardScaler. It scores both with 5-fold cross-validation and prints the mean accuracy, so you can compare the two geometries on equal footing.

In [ ]:
import matplotlib
import pandas as pd, numpy as np
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_score

df = pd.read_csv(r"iris.csv")
X = df[["sepal_length", "sepal_width", "petal_length", "petal_width"]]
y = df["species"]

for metric in ["euclidean", "manhattan"]:
    pipe = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5, metric=metric))
    scores = cross_val_score(pipe, X, y, cv=5)
    print(f"{metric:10s} 5-fold CV accuracy: {scores.mean():.4f}")


**What to notice:**
1. Both metrics give very high accuracy (mid-0.90s) — iris is a friendly dataset.
2. The two metrics rarely tie exactly; small differences come from how each weights large per-feature gaps.
3. Scaling is baked into the pipeline so both metrics see comparable features.

> **Pitfall:** KNN is a *lazy* learner — it stores all 150 rows and does all the work at prediction time. With 150 rows that's instant; with 10 million rows it would crawl.

> **In practice:** KNN powers "similar item" search in e-commerce, anomaly detection (a point far from everything else is suspicious), and imputation (fill a missing value with the average of its nearest neighbors). Production systems use approximate neighbor search (KD-trees, ball trees, FAISS) because brute-force distance to millions of rows is too slow.

---

## 2. Why scaling matters (and when it doesn't)

Imagine measuring height in centimeters and weight in kilograms, then computing distance without scaling. A 10 cm gap in height and a 10 kg gap in weight contribute the same raw amount to the distance — but heights range over ~30 cm while weights range over ~100 kg, so weight quietly dominates every neighbor search. Scaling is how you give every feature an equal vote.

Distance is measured in the units of your features. If one feature ranges 0–100 and another 0–1, the big-range feature dominates every distance calculation — the small-range feature might as well not exist.

A quick way to see this: compute each feature's **share of the total squared distance** (variance) across the dataset.

Here is the intuition: squared distance is additive across features. If feature A's values swing over a wide range, its squared gaps are huge, and they swamp the squared gaps of a feature that barely moves. KNN then effectively classifies on the wide-range feature alone — the other features are decoration.

The fix is StandardScaler: subtract each feature's mean and divide by its standard deviation. After scaling, every feature has mean 0 and std 1, so a "one unit" gap means the same thing — one standard deviation — for every feature. Distances become comparable, and every feature gets a say.

The snippet prints each feature's share of total squared distance, then runs a sabotage experiment: multiply `sepal_length` by 100 and watch unscaled KNN collapse. Finally it re-runs with StandardScaler in the pipeline to show the fix.

In [ ]:
import matplotlib
import pandas as pd, numpy as np
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_score

df = pd.read_csv(r"iris.csv")
X = df[["sepal_length", "sepal_width", "petal_length", "petal_width"]]
y = df["species"]

# Share of total squared distance contributed by each feature (unscaled)
sq = ((X - X.mean()) ** 2).mean()
shares = sq / sq.sum()
print("Unscaled squared-distance contribution:")
print(shares.round(4))

# Now blow up one feature x100 and watch KNN suffer
X_bad = X.copy()
X_bad["sepal_length"] = X_bad["sepal_length"] * 100
acc_bad = cross_val_score(KNeighborsClassifier(n_neighbors=5), X_bad, y, cv=5).mean()
print(f"\nUnscaled, sepal_length x100, k=5 CV accuracy: {acc_bad:.4f}")

# And the fix: scale it
acc_scaled = cross_val_score(
    make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5)), X, y, cv=5
).mean()
print(f"Scaled, k=5 CV accuracy: {acc_scaled:.4f}")


**What to notice:**
1. Unscaled, `petal_length` contributes ~68% of the squared distance — it drowns out the others.
2. On iris the damage is mild because all four features are already in centimeters (similar units). The x100 experiment shows what happens when units diverge: accuracy collapses.
3. `StandardScaler` (subtract mean, divide by std) gives every feature an equal vote.

> **Pitfall:** Scaling is not a free lunch — it assumes every feature *deserves* equal weight. If one feature is genuinely far more predictive, scaling can slightly dilute it. Always compare scaled vs unscaled with cross-validation, not with a single train/test split.

> **In practice:** Any distance- or gradient-based model — KNN, k-means, SVMs, neural networks, regularized regression — needs scaled features. Tree-based models (random forests, gradient boosting) are the exception: they split on single features, so scale never matters. A common production bug is scaling with statistics computed on the full dataset before the train/test split, which leaks test information into training.

---

## 3. Choosing k with cross-validation

k is the bias-variance trade-off wearing a small integer. Choose k badly and your model is either a nervous wreck that panics at every outlier, or a sleepy committee that always says "whatever the majority thinks". Cross-validation is how you find the sweet spot with data instead of vibes.

k is the one knob you must tune. k=1 memorizes the training set (one noisy neighbor decides everything). Huge k smooths into a blur where the majority class wins everywhere. The honest way to choose is a **cross-validation curve**: try many k values, estimate accuracy for each, pick the best.

With k=1, the prediction for any point is the label of its single closest training point. That makes the model hypersensitive: one mislabeled flower in the training set creates a tiny wrong-species island around itself. This is high variance — the model chases noise.

As k grows, each vote averages over more neighbors, and the decision smooths. But smoothing has a cost: the vote starts reaching across class boundaries, and in imbalanced data the majority class eventually wins everywhere. That is high bias — the model is too simple to see the structure.

The CV curve makes the trade-off visible. For each candidate k, cross-validation trains on 4/5 of the data and scores on the held-out 1/5, rotating five times, so every k gets an honest accuracy estimate that reflects performance on unseen flowers. The peak of the curve is your k.

The snippet sweeps k from 1 to 15, computing 5-fold CV accuracy for both scaled and unscaled pipelines at each k, prints the table, picks the best scaled k, and plots the two curves so you can see the shape of the trade-off.

In [ ]:
import matplotlib
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_score

df = pd.read_csv(r"iris.csv")
X = df[["sepal_length", "sepal_width", "petal_length", "petal_width"]]
y = df["species"]

ks = list(range(1, 16))
scaled_acc, unscaled_acc = [], []
for k in ks:
    s_scaled = cross_val_score(
        make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=k)), X, y, cv=5
    ).mean()
    s_raw = cross_val_score(KNeighborsClassifier(n_neighbors=k), X, y, cv=5).mean()
    scaled_acc.append(s_scaled)
    unscaled_acc.append(s_raw)
    print(f"k={k:2d}  scaled={s_scaled:.4f}  unscaled={s_raw:.4f}")

best_k = ks[int(np.argmax(scaled_acc))]
print(f"\nBest scaled k = {best_k} (CV accuracy {max(scaled_acc):.4f})")

plt.figure(figsize=(8, 4))
plt.plot(ks, scaled_acc, "o-", label="scaled")
plt.plot(ks, unscaled_acc, "s--", label="unscaled")
plt.xlabel("k (neighbors)")
plt.ylabel("5-fold CV accuracy")
plt.title("KNN accuracy vs k")
plt.legend()
plt.grid(alpha=0.3)
plt.savefig("knn_k_curve.png", dpi=100, bbox_inches="tight")
plt.show()


**What to notice:**
1. Accuracy is flat-to-noisy for small k, peaks in the middle (k≈5–8), then slowly declines as k grows.
2. On iris, unscaled edges out scaled — a reminder that scaling is a tool, not a religion.
3. The curve is the deliverable: it turns "what should k be?" from a guess into a measurement.

> **Pitfall:** Never pick k on the full dataset and then report that same number as your accuracy — that's training on the test set. The CV score is the honest estimate.

> **In practice:** The same curve method tunes every hyperparameter — tree depth, regularization strength, the number of clusters. The production version is `GridSearchCV` or `RandomizedSearchCV`, which automate the sweep and refit on the full training data with the winning setting. Always choose hyperparameters on validation data, never on the test set — the test set is for the final honest report.

---

## 4. Reading a decision boundary

A decision boundary is the map of "which region of feature-space belongs to which species". We can draw it by laying a fine grid over two features, asking the trained model to label every grid point, and coloring the regions. We use the two petal features because they separate the species best.

In [ ]:
import matplotlib
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import make_pipeline

df = pd.read_csv(r"iris.csv")
X = df[["petal_length", "petal_width"]]
y = df["species"]

le = LabelEncoder()
y_enc = le.fit_transform(y)
pipe = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5))
pipe.fit(X, y_enc)

# Build a grid over the two petal features
x_min, x_max = X["petal_length"].min() - 0.5, X["petal_length"].max() + 0.5
y_min, y_max = X["petal_width"].min() - 0.5, X["petal_width"].max() + 0.5
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200), np.linspace(y_min, y_max, 200))
grid = pd.DataFrame({"petal_length": xx.ravel(), "petal_width": yy.ravel()})
Z = pipe.predict(grid).reshape(xx.shape)

plt.figure(figsize=(8, 5))
plt.contourf(xx, yy, Z, alpha=0.3, cmap="viridis")
for species, marker in zip(["setosa", "versicolor", "virginica"], ["o", "s", "^"]):
    sub = df[df["species"] == species]
    plt.scatter(sub["petal_length"], sub["petal_width"], label=species, marker=marker, edgecolor="k")
plt.xlabel("petal_length")
plt.ylabel("petal_width")
plt.title("KNN (k=5) decision regions — petal features")
plt.legend()
plt.savefig("knn_decision_boundary.png", dpi=100, bbox_inches="tight")
plt.show()


**What to notice:**
1. Setosa is a clean island — it is linearly separable from the other two.
2. Versicolor and virginica share a fuzzy border; that is where the occasional misclassification lives.
3. The boundary is not a straight line — KNN carves piecewise regions around clusters of points.

> **Pitfall:** A 2D boundary plot is a *visualization* of a model that may use more features. Here we deliberately trained on only the two plotted features so the picture matches the model.

---

## Exercises (do these!)

### Exercise 1 — The k-curve
Using the code from section 3, record the scaled CV accuracy for every k from 1 to 15. Which k gives the best scaled accuracy, and what is that accuracy? Also note the best unscaled accuracy.
*Expected: best scaled k = 6 or 8 at CV accuracy ≈ 0.9667; best unscaled ≈ 0.9800 (k = 6, 7, 10, 11, or 12).*

**Follow-up:** Why does accuracy dip at k=1 and again for large k? Check: k=1 scaled ≈ 0.9467, k=15 scaled ≈ 0.9400.

<details>
<summary>Hint</summary>

`np.argmax(scaled_acc)` gives the index of the best k; add 1 because k starts at 1.

</details>

### Exercise 2 — The scale trap
Compute each feature's share of total squared distance (section 2). Which feature dominates? Then multiply `sepal_length` by 100 and re-run 5-fold CV with k=5 unscaled. How much accuracy do you lose?
*Expected: petal_length dominates with ≈ 0.68 of squared distance; the x100 experiment drops CV accuracy to ≈ 0.7333 (from ≈ 0.96).*

**Follow-up:** What does `StandardScaler` do to each feature's mean and std? Check: after scaling, every feature has mean ≈ 0 and std ≈ 1.

<details>
<summary>Hint</summary>

`StandardScaler().fit(X).transform(X)` returns a numpy array; wrap it in a DataFrame to inspect `.mean()` and `.std()`.

</details>

### Exercise 3 — Classify a hand-entered flower
Train a scaled KNN (k=5) on all four features. Predict the species of a flower with sepal_length=6.0, sepal_width=2.9, petal_length=4.5, petal_width=1.5. What is the predicted species and the class probabilities?
*Expected: predicted species = versicolor, probabilities ≈ [0.0, 1.0, 0.0] (setosa, versicolor, virginica).*

**Follow-up:** Move petal_length to 5.5 and keep everything else. What changes? Check: the prediction flips to virginica.

<details>
<summary>Hint</summary>

Build a DataFrame with the same column names as X, then call `pipe.predict_proba(flower)`. The columns of `pipe.classes_` give the probability order.

</details>

---

## Solutions

Try for 15 min each before peeking.

In [ ]:
# --- Solution 1 ---
import matplotlib
import pandas as pd, numpy as np
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_score

df = pd.read_csv(r"iris.csv")
X = df[["sepal_length", "sepal_width", "petal_length", "petal_width"]]
y = df["species"]

ks = list(range(1, 16))
scaled_acc = [cross_val_score(make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=k)), X, y, cv=5).mean() for k in ks]
unscaled_acc = [cross_val_score(KNeighborsClassifier(n_neighbors=k), X, y, cv=5).mean() for k in ks]
best_k = ks[int(np.argmax(scaled_acc))]
print(f"best scaled k={best_k}, acc={max(scaled_acc):.4f}")
print(f"best unscaled acc={max(unscaled_acc):.4f}")

# --- Solution 2 ---
import matplotlib
import pandas as pd, numpy as np
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_score

df = pd.read_csv(r"iris.csv")
X = df[["sepal_length", "sepal_width", "petal_length", "petal_width"]]
y = df["species"]

sq = ((X - X.mean()) ** 2).mean()
print("distance shares:")
print((sq / sq.sum()).round(4))

X_bad = X.copy()
X_bad["sepal_length"] = X_bad["sepal_length"] * 100
acc_bad = cross_val_score(KNeighborsClassifier(n_neighbors=5), X_bad, y, cv=5).mean()
print(f"x100 unscaled k=5 CV acc: {acc_bad:.4f}")

# --- Solution 3 ---
import matplotlib
import pandas as pd, numpy as np
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

df = pd.read_csv(r"iris.csv")
X = df[["sepal_length", "sepal_width", "petal_length", "petal_width"]]
y = df["species"]

pipe = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5))
pipe.fit(X, y)
flower = pd.DataFrame({"sepal_length": [6.0], "sepal_width": [2.9], "petal_length": [4.5], "petal_width": [1.5]})
print("prediction:", pipe.predict(flower)[0])
print("probabilities:", pipe.predict_proba(flower).round(3))
print("classes:", pipe.classes_)

# --- Follow-up 1 ---
import matplotlib
import pandas as pd, numpy as np
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_score

df = pd.read_csv(r"iris.csv")
X = df[["sepal_length", "sepal_width", "petal_length", "petal_width"]]
y = df["species"]
acc1 = cross_val_score(make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=1)), X, y, cv=5).mean()
acc15 = cross_val_score(make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=15)), X, y, cv=5).mean()
assert abs(acc1 - 0.9467) < 0.01, acc1
assert abs(acc15 - 0.9400) < 0.01, acc15
print(f"k=1: {acc1:.4f}, k=15: {acc15:.4f} — both worse than the middle")

# --- Follow-up 2 ---
import matplotlib
import pandas as pd, numpy as np
from sklearn.preprocessing import StandardScaler

df = pd.read_csv(r"iris.csv")
X = df[["sepal_length", "sepal_width", "petal_length", "petal_width"]]
Xs = pd.DataFrame(StandardScaler().fit_transform(X), columns=X.columns)
assert np.allclose(Xs.mean(), 0, atol=1e-9)
assert np.allclose(Xs.std(ddof=0), 1, atol=1e-9)
print("scaled means ~ 0, scaled stds ~ 1: OK")

# --- Follow-up 3 ---
import matplotlib
import pandas as pd, numpy as np
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

df = pd.read_csv(r"iris.csv")
X = df[["sepal_length", "sepal_width", "petal_length", "petal_width"]]
y = df["species"]
pipe = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5))
pipe.fit(X, y)
flower2 = pd.DataFrame({"sepal_length": [6.0], "sepal_width": [2.9], "petal_length": [5.5], "petal_width": [1.5]})
pred2 = pipe.predict(flower2)[0]
assert pred2 == "virginica", pred2
print("petal_length=5.5 ->", pred2)


### What to learn next
- KNN regression (predict a number, not a label, by averaging neighbors).
- The curse of dimensionality: distances get meaningless in very high-dimensional spaces.
- Decision trees — a model that carves the same kind of regions with axis-aligned rules.

*Files in this folder: iris.csv (150 rows: sepal_length, sepal_width, petal_length, petal_width, species).*

---

**Done with Colab?** Download the notebook (**File → Download .ipynb**) to keep outputs, or **File → Save a copy in Drive**. Re-upload datasets after a runtime recycle.
